# TP 0 — Understanding Bagging and Random Forests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session2/tp_0_bagging.ipynb)

## Objectives
1. Understand why single decision trees suffer from high variance
2. Learn how bootstrap sampling works
3. See how Bagging reduces variance by averaging many trees
4. Understand Random Forests as Bagging with feature randomization
5. Use Out-of-Bag estimation and feature importances

## Setup

Run the cell below to install and import the required packages.

In [ ]:
!pip install git+https://github.com/racousin/L2Math.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

from l2math import plot_decision_boundary_2d

np.random.seed(42)
print("Setup complete!")

---
# Part 1: The Problem — High Variance

A single decision tree is an **unstable** learner: small changes in the training data can produce a completely different tree structure and decision boundary.

This is the hallmark of a **high-variance** model. Let's see this in action by training four decision trees on slightly different bootstrap samples drawn from the same dataset.

In [ ]:
# Generate a noisy two-class dataset
X, y = make_moons(n_samples=200, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train 4 trees on 4 different bootstrap samples of the training data
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for i, ax in enumerate(axes.ravel()):
    # Draw a bootstrap sample (same size as training set, with replacement)
    X_boot, y_boot = resample(X_train, y_train, random_state=i * 10)
    
    tree = DecisionTreeClassifier(random_state=i)
    tree.fit(X_boot, y_boot)
    acc = accuracy_score(y_test, tree.predict(X_test))
    
    plot_decision_boundary_2d(tree, X_train, y_train,
                              title=f"Tree {i+1} (bootstrap sample, test acc={acc:.2f})", ax=ax)

plt.suptitle("4 Decision Trees on Different Bootstrap Samples", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Notice how different each decision boundary looks!")
print("This instability is exactly the high-variance problem.")

---
# Part 2: Bootstrap Sampling

A **bootstrap sample** is obtained by drawing $n$ samples **with replacement** from a dataset of size $n$.

Because sampling is done with replacement, some data points will appear **multiple times** and others will be **left out** entirely.

In [ ]:
# Illustration with a tiny dataset of 10 elements
original = np.arange(10)
print("Original dataset:", original)
print("=" * 60)

for s in range(3):
    np.random.seed(s)
    boot = np.random.choice(original, size=len(original), replace=True)
    boot_sorted = np.sort(boot)
    
    # Count occurrences
    counts = np.bincount(boot, minlength=len(original))
    
    missing = original[counts == 0]
    once = original[counts == 1]
    multi = original[counts >= 2]
    
    print(f"\nBootstrap sample {s+1}: {boot_sorted}")
    print(f"  Absent  (0 times): {missing}")
    print(f"  Once    (1 time) : {once}")
    print(f"  Repeated (2+ times): {multi}")
    print(f"  Unique elements: {len(np.unique(boot))}/{len(original)}")

### How much data appears in each bootstrap sample?

The probability that a given element is **not** selected in any single draw is $1 - \frac{1}{n}$. After $n$ draws with replacement:

$$P(\text{element absent}) = \left(1 - \frac{1}{n}\right)^n \xrightarrow{n \to \infty} \frac{1}{e} \approx 0.368$$

So on average, each bootstrap sample contains about $1 - 1/e \approx 63.2\%$ of the unique data points.

Let's verify this empirically.

In [ ]:
# Empirical verification with large n
n = 10000
n_bootstrap_samples = 200

fractions = []
for i in range(n_bootstrap_samples):
    boot = np.random.choice(n, size=n, replace=True)
    frac_unique = len(np.unique(boot)) / n
    fractions.append(frac_unique)

theoretical = 1 - 1 / np.e
empirical = np.mean(fractions)

print(f"Dataset size n = {n}")
print(f"Number of bootstrap samples = {n_bootstrap_samples}")
print(f"Theoretical fraction of unique elements: {theoretical:.4f}")
print(f"Empirical   fraction of unique elements: {empirical:.4f}")
print(f"Difference: {abs(theoretical - empirical):.4f}")

---
# Part 3: Bagging (Bootstrap Aggregating)

**Bagging** exploits the instability of decision trees as a feature, not a bug:

1. Draw $B$ bootstrap samples from the training data
2. Train one decision tree on each bootstrap sample
3. **Aggregate** predictions: majority vote (classification) or average (regression)

### Why does this reduce variance?

If the $B$ trees were truly independent with variance $\sigma^2$, then the variance of their average would be:

$$\text{Var}(\bar{f}) = \frac{1}{B}\sigma^2$$

In practice the trees are correlated (they use the same training data), so the reduction is not $1/B$, but averaging still helps significantly.

In [ ]:
# 1D Regression: Bagging smooths out noisy individual trees
np.random.seed(42)

# Generate noisy sine data
X_reg = np.sort(np.random.uniform(0, 2 * np.pi, 80)).reshape(-1, 1)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.3, X_reg.shape[0])

# Fine grid for plotting
X_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)

# Train individual trees on bootstrap samples and collect predictions
n_trees = 10
all_preds = []

fig, ax = plt.subplots(figsize=(10, 5))

for i in range(n_trees):
    X_boot, y_boot = resample(X_reg, y_reg, random_state=i)
    tree = DecisionTreeRegressor(max_depth=5, random_state=i)
    tree.fit(X_boot, y_boot)
    pred = tree.predict(X_plot)
    all_preds.append(pred)
    ax.plot(X_plot, pred, color='gray', alpha=0.25, linewidth=1,
            label='Individual trees' if i == 0 else None)

# Bagged average
bagged_pred = np.mean(all_preds, axis=0)
ax.plot(X_plot, bagged_pred, color='red', linewidth=2.5, label='Bagged average')

# True function
ax.plot(X_plot, np.sin(X_plot), color='blue', linewidth=2, linestyle='--', label='True: sin(x)')

# Training points
ax.scatter(X_reg, y_reg, color='black', s=15, zorder=5, label='Training data')

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Bagging Regression: Individual Trees vs Averaged Prediction', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The red bagged average is much smoother than any individual gray tree.")

---
# Part 4: Bagging in Action — Classification

Let's compare the decision boundary of a single decision tree with a `BaggingClassifier` using 50 trees.

In [ ]:
# Use the same make_moons dataset
X, y = make_moons(n_samples=200, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Single decision tree
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
acc_tree = accuracy_score(y_test, single_tree.predict(X_test))

# Bagging classifier
bagging = BaggingClassifier(n_estimators=50, random_state=42)
bagging.fit(X_train, y_train)
acc_bag = accuracy_score(y_test, bagging.predict(X_test))

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_decision_boundary_2d(single_tree, X_train, y_train,
                          title=f"Single Decision Tree (test acc={acc_tree:.2f})", ax=axes[0])
plot_decision_boundary_2d(bagging, X_train, y_train,
                          title=f"Bagging — 50 Trees (test acc={acc_bag:.2f})", ax=axes[1])

plt.tight_layout()
plt.show()

print(f"Single tree test accuracy: {acc_tree:.3f}")
print(f"Bagging test accuracy:     {acc_bag:.3f}")
print("\nBagging produces a much smoother, more generalizable boundary.")

---
# Part 5: Accuracy vs Number of Estimators

A key practical question: **how many trees do we need?**

More trees never hurt (no overfitting from adding trees), but eventually the gains saturate.

In [ ]:
n_estimators_range = list(range(1, 101))
train_accuracies = []
test_accuracies = []

for n_est in n_estimators_range:
    bag = BaggingClassifier(n_estimators=n_est, random_state=42)
    bag.fit(X_train, y_train)
    train_accuracies.append(accuracy_score(y_train, bag.predict(X_train)))
    test_accuracies.append(accuracy_score(y_test, bag.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_estimators_range, train_accuracies, label='Train accuracy', linewidth=2)
ax.plot(n_estimators_range, test_accuracies, label='Test accuracy', linewidth=2)
ax.set_xlabel('Number of estimators (B)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Bagging: Accuracy vs Number of Trees', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best test accuracy: {max(test_accuracies):.3f} at B={n_estimators_range[np.argmax(test_accuracies)]}")
print("Test accuracy saturates — more trees don't hurt but eventually stop helping.")

---
# Part 6: Random Forest = Bagging + Feature Randomization

### Why add more randomness?

In standard bagging, if one feature is very strong, **all trees will split on it first**, making the trees highly correlated. Correlated trees means less variance reduction.

The true variance of the bagged average when trees have pairwise correlation $\rho$ is:

$$\text{Var}(\bar{f}) = \rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$$

The first term $\rho\sigma^2$ does not vanish with $B$. The only way to reduce it is to **decrease the correlation** $\rho$ between trees.

### Random Forest solution

At each split, consider only a **random subset** of $\sqrt{p}$ features (classification) or $p/3$ features (regression), where $p$ is the total number of features. This decorrelates the trees and further reduces variance.

In [ ]:
# Generate a dataset with 10 features (only 5 informative)
X_hd, y_hd = make_classification(
    n_samples=300, n_features=10, n_informative=5,
    n_redundant=2, n_clusters_per_class=2, random_state=42
)

X_hd_train, X_hd_test, y_hd_train, y_hd_test = train_test_split(
    X_hd, y_hd, test_size=0.3, random_state=42
)

# For 2D visualization, use only the first 2 features
X_vis_train = X_hd_train[:, :2]
X_vis_test = X_hd_test[:, :2]

# BaggingClassifier on 2 features
bag_2d = BaggingClassifier(n_estimators=50, random_state=42)
bag_2d.fit(X_vis_train, y_hd_train)
acc_bag_2d = accuracy_score(y_hd_test, bag_2d.predict(X_vis_test))

# RandomForestClassifier on 2 features
rf_2d = RandomForestClassifier(n_estimators=50, random_state=42)
rf_2d.fit(X_vis_train, y_hd_train)
acc_rf_2d = accuracy_score(y_hd_test, rf_2d.predict(X_vis_test))

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_decision_boundary_2d(bag_2d, X_vis_train, y_hd_train,
                          title=f"BaggingClassifier (test acc={acc_bag_2d:.2f})", ax=axes[0])
plot_decision_boundary_2d(rf_2d, X_vis_train, y_hd_train,
                          title=f"RandomForestClassifier (test acc={acc_rf_2d:.2f})", ax=axes[1])

plt.tight_layout()
plt.show()

print(f"Bagging test accuracy (2 features):        {acc_bag_2d:.3f}")
print(f"Random Forest test accuracy (2 features):  {acc_rf_2d:.3f}")
print("\nWith only 2 features the difference is subtle.")
print("The real benefit of feature randomization shines with many features.")

---
# Part 7: Out-of-Bag (OOB) Estimation

Recall that each bootstrap sample leaves out about 36.8% of the training data. For each training point, we can collect predictions from **only the trees that did not see it** during training. This gives a "free" validation estimate called the **Out-of-Bag (OOB) score**.

No need for a separate validation set!

In [ ]:
# Train a Random Forest with OOB scoring enabled
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf_oob.fit(X_train, y_train)

oob_score = rf_oob.oob_score_
test_score = accuracy_score(y_test, rf_oob.predict(X_test))

print("=" * 45)
print(" Out-of-Bag vs Test Accuracy")
print("=" * 45)
print(f"  OOB  accuracy: {oob_score:.4f}")
print(f"  Test accuracy: {test_score:.4f}")
print(f"  Difference:    {abs(oob_score - test_score):.4f}")
print("=" * 45)
print("\nThe OOB score is a reliable estimate of test performance.")

---
# Part 8: Feature Importance in Random Forests

Random Forests provide a natural measure of **feature importance** based on how much each feature reduces impurity (e.g., Gini index) across all trees.

We use the 10-feature dataset from Part 6 (with only 5 truly informative features) to see if the model can identify which features matter.

In [ ]:
# Train a Random Forest on all 10 features
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_hd_train, y_hd_train)

acc_full = accuracy_score(y_hd_test, rf_full.predict(X_hd_test))
print(f"Random Forest accuracy (all 10 features): {acc_full:.3f}\n")

# Extract and plot feature importances
importances = rf_full.feature_importances_
feature_names = [f"Feature {i}" for i in range(X_hd.shape[1])]

# Sort by importance
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color='steelblue', edgecolor='k')
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Random Forest Feature Importances', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("The dataset was generated with 5 informative features.")
print("The Random Forest correctly assigns higher importance to the useful features.")

---
# Summary

## From Bagging to Random Forests

| Method | Idea | Key Benefit |
|--------|------|-------------|
| **Single Tree** | One tree on full data | Simple, interpretable, but high variance |
| **Bagging** | $B$ trees on $B$ bootstrap samples, majority vote | Reduces variance via averaging |
| **Random Forest** | Bagging + random feature subsets at each split | Decorrelates trees, further reduces variance |

## Key Hyperparameters

- `n_estimators` ($B$): number of trees. More is better up to a point; no overfitting risk from adding more.
- `max_features`: number of features considered at each split. $\sqrt{p}$ for classification, $p/3$ for regression.
- `max_depth`: maximum depth of each tree. Deeper trees = less bias, more variance per tree.

## Pros and Cons

**Pros:**
- Very effective out of the box with minimal tuning
- Built-in feature importance
- Out-of-Bag estimation (free cross-validation)
- Robust to outliers and noise
- Handles both classification and regression

**Cons:**
- Less interpretable than a single tree
- Can be slow to train with many trees and large datasets
- Does not extrapolate well (same limitation as decision trees)
- Tends to underperform gradient boosting on structured/tabular data (we will see this next!)